### environment

In [ ]:
%pip install -U spacy spacy-transformers
%pip install -U spacy
%pip install --upgrade pip
%pip install pip setuptools wheel
!python -m spacy download en_core_web_sm 

In [1]:
import os, rich, json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import spacy
from spacy.tokens import DocBin, Doc
from label_models import Label

In [ ]:
# spacy.prefer_gpu()

In [2]:
nlp = spacy.load("en_core_web_sm") 
doc_bin = DocBin()

/home/ronji/repos/reverse-ats/ner-spacy-dev/ner-spacy/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### processing

In [3]:
def split_doc(doc, labels, MAX_TOKENS=500, MIN_TOKENS = 10): 
    sequences = []
    current_start = 0 
    '''
    split a doc object into sequences short enough (MAX_TOKENS) to be used in spacy's model training
    MIN_TOKENS: to create reasonably long sequences in case of last few remaining tokens
    '''
    while current_start < len(doc):
        current_end = min(current_start + MAX_TOKENS, len(doc))
        subdoc_tokens = doc[current_start:current_end]  # Get tokens for the chunk to allow slicing
        subdoc = Doc(doc.vocab, words=[token.text for token in subdoc_tokens])  # Create a new Doc object
        
        subdoc_start_char = subdoc_tokens[0].idx 
        subdoc_end_char = subdoc_tokens[-1].idx + len(subdoc_tokens[-1])
        
        seq_labels = [
            Label(
                label['start'] - subdoc_start_char,
                label['end'] - subdoc_start_char,
                label['label'],
                label['value']
            )
            for label in labels # iterate labels only in the current chunk
            if label["start"] >= subdoc_start_char and label["end"] <= subdoc_end_char
        ]
        if len(subdoc) > MIN_TOKENS or len(seq_labels) > 0:
            sequences.append((subdoc, seq_labels))
        current_start = current_end
    return sequences


In [4]:
def remove_whitespace(content, labels):
    leading = len(content) - len(content.lstrip())
    content_no_ws = content.strip()
    new_labels = []
    for label in labels:
        adjusted_start = label["start"] - leading
        adjusted_end = label["end"] - leading
        if adjusted_start >= 0 and adjusted_end <= len(content_no_ws):
            new_labels.append({
                "start": adjusted_start,
                "end": adjusted_end,
                "label": label["label"],
                "value": label["value"]
            })
    return content_no_ws, new_labels

In [5]:
with open(f'{os.getcwd()}/labels_en.json','r') as f:
    dataset = json.loads(f.read())
    train, test = train_test_split(dataset, test_size=0.2, random_state=42)
    with open('test.json','w') as jf:
        jf.write(json.dumps(test))
    

doc_bin = DocBin()

empty_span_count, overlap_count, ent_count = 0,0,0 

for doc in tqdm(train):
    source_id, content, labels = doc['id'], doc['content'], doc['labels']
    content, labels = remove_whitespace(content, labels)
    doc = nlp(content)
    sequences = split_doc(doc,labels)

    for seq_doc, seq_labels in sequences:
        ents = []
        for l in seq_labels:
            start, end, label, value = l.start, l.end, l.label, l.value
            span = seq_doc.char_span(start, end, label, alignment_mode='expand')
            if span is None:
                empty_span_count += 1
                continue
            if any(
                    span.start < existing_span.end and 
                    span.end > existing_span.start 
                for existing_span in ents):
                overlap_count += 1 #check overlapping spand caused by alignment_mode in some cases
                continue
            ents.append(span)
            if len(ents) > 0: ent_count += len(ents) 
        if len(ents) > 0: # filter out (mostly short) docs with no valid entities
            seq_doc.set_ents(ents)  
            doc_bin.add(seq_doc)

doc_bin.to_disk("train.spacy")
print(f'final entities: {ent_count}\noverlaps: {overlap_count}\nempty spans:{empty_span_count}')
# after trying multiple ways of processing the data, this proved to be the most effective for entity count


100%|██████████| 1964/1964 [03:11<00:00, 10.25it/s]


final entities: 335090
overlaps: 6682
empty spans:0


In [6]:
# check if data saved correctly
doc_count = 0
no_entities = 0
doc_bin = DocBin().from_disk("train.spacy")
for doc in doc_bin.get_docs(nlp.vocab):
    doc_count += 1
    if len(doc.ents) == 0: no_entities += 1

print(doc_count)
print(no_entities)


3344
0
